In [1]:
from torch_geometric.datasets import GNNBenchmarkDataset

dataset = GNNBenchmarkDataset(root="../../data/", name="MNIST").to("cuda:0")

Extracting ../../data/MNIST/raw/MNIST_v2.zip
Processing...
/home/jonathan/projects/primaite/PrimAITE/venv/lib/python3.10/site-packages/torch_geometric/datasets/gnn_benchmark_dataset.py:184: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on

In [29]:
train_test_split = 0.75
split_idx = int(train_test_split * len(dataset))

train_dataset = dataset[:split_idx]
test_dataset = dataset[split_idx:]

In [79]:
import networkx as nx

def generate_instructions(G):
    num_nodes = G.number_of_nodes()
    
    instructions = [
        "Find the node with the highest degree in the graph.",
        "Count the number of edges in the graph.",
        "Is the graph connected?",
        f"What is the shortest path between node 0 and node {num_nodes-1}?",
        "How many triangles are in the graph?"
    ]
    
    answers = [
        f"Node {max(G.degree, key=lambda x: x[1])[0]} has the highest degree.",
        f"The graph has {G.number_of_edges()} edges.",
        f"The graph is {'connected' if nx.is_connected(G) else 'not connected'}.",
        f"The shortest path is {nx.shortest_path(G, 0, num_nodes-1)}.",
        f"There are {sum(nx.triangles(G).values()) // 3} triangles in the graph."
    ]
    
    return list(zip(instructions, answers))

In [80]:
from torch_geometric.loader import DataLoader

train_dataloader = DataLoader(dataset=train_dataset, batch_size=1)
test_dataloader = DataLoader(dataset=test_dataset, batch_size=64)

In [81]:
from torch_geometric.utils import to_networkx

graphs = []
for graph in train_dataloader:
    G = to_networkx(graph)
    grap
    instructions = generate_instructions(nx.Graph(G))
    print(instructions)
    break

[('Find the node with the highest degree in the graph.', 'Node 17 has the highest degree.'), ('Count the number of edges in the graph.', 'The graph has 334 edges.'), ('Is the graph connected?', 'The graph is connected.'), ('What is the shortest path between node 0 and node 68?', 'The shortest path is [0, 48, 6, 68].'), ('How many triangles are in the graph?', 'There are 553 triangles in the graph.')]


In [83]:
from torch_geometric.data import Dataset


class InstructionTuning(Dataset):
    def __init__(self, graphs, instruction_pairs):
        assert len(graphs) == len(instruction_pairs)
        self.graphs = graphs
        self.instruction_pairs = instruction_pairs
        
    def __len__(self) -> int:
        return len(self.graphs)
    
    def __getitem__(self, idx):
        return (self.graphs[idx], self.instruction_pairs[idx])

In [ ]:
dataset = InstructionTuning(graphs=)

In [31]:
from torch import nn
import torch.nn.functional as F
from torch_geometric.nn import GATConv
from torch_geometric.nn.pool import global_mean_pool

class GraphEmbedding(nn.Module):
    def __init__(self,in_channels, hidden_channels, out_channels):
        super(GraphEmbedding, self).__init__()
        self.conv1 = GATConv(in_channels, hidden_channels)
        self.conv2 = GATConv(hidden_channels, hidden_channels)
        self.conv3 = GATConv(hidden_channels, hidden_channels)
        self.lin = nn.Linear(hidden_channels, out_channels)

    def forward(self, x, edge_index, batch):
        # 1. Obtain node embeddings 
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = self.conv2(x, edge_index)
        x = F.relu(x)
        x = self.conv3(x, edge_index)

        # 2. Readout layer
        x = global_mean_pool(x, batch)  # [batch_size, hidden_channels]

        x = F.dropout(x, p=0.5, training=self.training)
        x = self.lin(x)
        
        return x

In [169]:
import torch
from transformers import BertModel, BertTokenizer, LlamaForCausalLM, AutoTokenizer

class TextEncoder(nn.Module):
    def __init__(self, model_name: str = "HuggingFaceTB/SmolLM-1.7B-Instruct"):
        super().__init__()
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = LlamaForCausalLM.from_pretrained(model_name)

    def forward(self, x, edge_index, batch):
        prompts = f"Node features:\n{x}\n\n"
        
        encoded = self.tokenizer.encode(
            prompts,
            padding=True,
            truncation=True,
            max_length=512,
            return_tensors="pt"
        ).to("cuda:0")
        
        outputs = self.model(encoded)
        return outputs.last_hidden_state[:,0,:]

In [145]:
from transformers import AutoTokenizer, LlamaForCausalLM

tokenizer = AutoTokenizer.from_pretrained("HuggingFaceTB/SmolLM-1.7B-Instruct")
text_prediction_model = LlamaForCausalLM.from_pretrained("HuggingFaceTB/SmolLM-1.7B-Instruct").to("cuda:0")


In [166]:
messages = [{"role": "user", "content": "what nosie does a fox make"}]
tokens=tokenizer.apply_chat_template(messages, tokenize=True, return_tensors="pt").to("cuda:0")
output = text_prediction_model.generate(tokens, max_new_tokens=1024)
response = tokenizer.decode(output[0])
print(response)

<|im_start|>user
what nosie does a fox make<|im_end|>
<|im_start|>assistant
The fox's noise is a combination of a bark and a whimper. The bark is a high-pitched sound that is typically associated with aggression or fear, while the whimper is a low-pitched sound that is often used to communicate distress or submission. The fox's bark is likely a response to the presence of a predator, such as a dog or human, while the whimper may be a way to signal that the fox is in trouble.<|im_end|>


In [170]:
gnn = GraphEmbedding(in_channels=dataset.num_features, out_channels=768, hidden_channels=128).to("cuda:0")
llm = TextEncoder().to("cuda:0")

OutOfMemoryError: CUDA out of memory. Tried to allocate 16.00 MiB. GPU 

In [1]:
class AlignmentProjector(nn.Module):
    def __init__(self, graph_embedding_dim, text_embedding_dim, output_dim, hidden_dim):
        self.linear1 = nn.Linear(in_features=graph_embedding_dim+text_embedding_dim, out_features=hidden_dim)
        self.linear2 = nn.Linear(in_features=hidden_dim, out_features=output_dim)
        
    def forward(self, x):
        x = F.relu(self.linear1(x))
        x = F.relu(self.linear2(x))
        x = F.dropout(x, p=0.2)
        return x

NameError: name 'nn' is not defined

In [34]:
def test(test_data, gnn, te, criterion):
    gnn.eval()
    test_loss = 0

    for data in test_data:
        with torch.no_grad():
            graph_output = gnn(data.x, data.edge_index, data.batch)
            text_output = te(data.x, data.edge_index, data.batch)

            loss = criterion(text_output, graph_output, torch.ones(1).to("cuda:0"))
            test_loss += loss.item()

    return test_loss / len(test_data)

In [35]:
def train(train_data, gnn, te, criterion, opt):
    gnn.train()
    epoch_loss = 0
    for data in train_data:

        graph_output = gnn(data.x, data.edge_index, data.batch)
        text_output = te(data.x, data.edge_index, data.batch)
        
        loss = criterion(text_output, graph_output, torch.ones(1).to("cuda:0"))
        epoch_loss += loss.item()
        loss.backward()
        opt.step()

    return epoch_loss / len(train_data)

In [36]:

class AlignmentProjector(nn.Module):
    def __init__(self, input_dim, output_dim):
        self.linear1 = nn.Linear(input_dim, input_dim * 2)
        self.linear2 = nn.Linear(input_dim * 2, output_dim)
        
    def forward(self, x):
        x = self.linear1(x)
        x = F.relu(x)
        x = self.linear2(x)
        x = F.dropout(x, p=0.2)
        return F.relu(x)

In [ ]:
# from torch.optim import Adam
# import matplotlib.pyplot as plt

# criterion = nn.CosineEmbeddingLoss()
# optimizer = Adam(gnn.parameters(), lr=0.001)

# train_losses = []
# test_losses = []

# for epoch in range(3):
#     print(f"====== Epoch {epoch + 1} =======")
#     train_loss = train(train_dataloader, gnn, llm, criterion, optimizer)
#     train_losses.append(train_loss)
#     test_loss = test(test_dataloader, gnn, llm, criterion)
#     test_losses.append(test_loss)
#     print(f"Train Loss: {train_loss}, Test Loss = {test_loss}\n")
    
# torch.save(gnn.state_dict(), "model.pt")

# plt.plot(range(len(train_losses)), train_losses, label="Train")
# plt.plot(range(len(test_losses)), test_losses, label="Test")
# plt.xlabel("Epochs")
# plt.ylabel("Loss")
# plt.legend()
# plt.savefig("learning_curve.pdf", format="pdf")